# 🎲 Randomized Behavior

**One algorithm — QuickSort with random pivots — to understand why randomness is a powerful tool in algorithm design.**

---

## 1. Concept Intuition

### The QuickSort story

QuickSort works by picking a **pivot**, then splitting the array: elements smaller than the pivot go left, larger go right. Repeat recursively.

**The catch:** if you always pick the worst pivot (smallest or largest element), one side gets *everything* and the other gets *nothing*. That's n levels of recursion, each doing n work → O(n²).

**The fix:** pick the pivot **randomly**. Now the adversary can't force the worst case, because they don't control your coin.

### Probability intuition

Think of expected value like a **center of gravity**. If you rolled a die a million times, the average result would be ~3.5. Not because 3.5 is a possible outcome, but because it's where all outcomes *balance*.

For randomized QuickSort:
- Each run might be fast or slow
- But the **expected** number of comparisons across all possible random choices is O(n log n)
- And this holds for *any* input — the randomness is in the algorithm, not the data

## 2. Visual Representation

Let's compare deterministic (worst-case) vs randomized QuickSort.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
%matplotlib inline

def quicksort_count(arr, random_pivot=True):
    """QuickSort that counts comparisons."""
    comparisons = [0]
    
    def _sort(a):
        if len(a) <= 1:
            return a
        if random_pivot:
            pivot_idx = np.random.randint(len(a))
        else:
            pivot_idx = 0  # First element — worst case on sorted input
        pivot = a[pivot_idx]
        left = [x for x in a if x < pivot]
        middle = [x for x in a if x == pivot]
        right = [x for x in a if x > pivot]
        comparisons[0] += len(a) - 1
        return _sort(left) + middle + _sort(right)
    
    _sort(list(arr))
    return comparisons[0]

# Worst case: sorted input with first-element pivot
sizes = np.arange(10, 310, 10)
worst_case = [quicksort_count(list(range(n)), random_pivot=False) for n in sizes]

# Average case: sorted input with random pivot (single run)
random_case = [quicksort_count(list(range(n)), random_pivot=True) for n in sizes]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes, worst_case, 'o-', color='#f43f5e', label='Deterministic (first pivot) → O(n²)', 
        linewidth=2, markersize=4)
ax.plot(sizes, random_case, 's-', color='#10b981', label='Randomized pivot → ~O(n log n)', 
        linewidth=2, markersize=4)

# Theoretical curves
n_theory = np.linspace(10, 300, 200)
ax.plot(n_theory, n_theory**2 / 2, '--', color='#f43f5e', alpha=0.3, label='n²/2 (theory)')
ax.plot(n_theory, 2 * n_theory * np.log2(n_theory), '--', color='#10b981', alpha=0.3, 
        label='2n·log₂n (theory)')

ax.set_xlabel('n (input size)', fontsize=12)
ax.set_ylabel('Comparisons', fontsize=12)
ax.set_title('QuickSort: Worst Case vs Randomized', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Mathematical Formulation

### Expected Value

For a discrete random variable $X$:

$$E[X] = \sum_{x} x \cdot P(X = x)$$

It's the probability-weighted average of all possible outcomes.

### QuickSort Expected Comparisons

For randomized QuickSort on $n$ elements, define $X_{ij}$ = 1 if elements $i$ and $j$ are ever compared.

$$E[\text{total comparisons}] = \sum_{i < j} P(i \text{ and } j \text{ are compared})$$

Two elements $z_i$ and $z_j$ (in sorted order) are compared **only if one of them is chosen as a pivot before any element between them**. This probability is:

$$P(z_i \text{ compared with } z_j) = \frac{2}{j - i + 1}$$

Summing this over all pairs:

$$E[\text{comparisons}] = \sum_{i=1}^{n} \sum_{j=i+1}^{n} \frac{2}{j - i + 1} = 2n \cdot H_n - 2n \approx 2n \ln n = O(n \log n)$$

where $H_n = 1 + \frac{1}{2} + \frac{1}{3} + \cdots + \frac{1}{n}$ is the Harmonic number.

## 4. Code Implementation

### Monte Carlo: Empirically measuring expected comparisons

In [ ]:
def monte_carlo_quicksort(n, trials=500):
    """Run randomized QuickSort many times and track comparison distribution."""
    arr = list(range(n))  # Worst case for deterministic — doesn't matter for random
    counts = [quicksort_count(arr, random_pivot=True) for _ in range(trials)]
    return np.array(counts)

n = 200
comparisons = monte_carlo_quicksort(n, trials=1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of comparisons
axes[0].hist(comparisons, bins=40, color='#6366f1', alpha=0.7, edgecolor='white')
axes[0].axvline(comparisons.mean(), color='#f97316', linestyle='--', linewidth=2,
                label=f'Mean = {comparisons.mean():.0f}')
axes[0].axvline(2 * n * np.log(n), color='#10b981', linestyle=':', linewidth=2,
                label=f'2n·ln(n) = {2*n*np.log(n):.0f}')
axes[0].set_title(f'Distribution of comparisons (n={n}, 1000 trials)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Comparisons')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Running average converges
running_avg = np.cumsum(comparisons) / np.arange(1, len(comparisons) + 1)
axes[1].plot(running_avg, color='#6366f1', linewidth=1.5)
axes[1].axhline(2 * n * np.log(n), color='#10b981', linestyle=':', linewidth=2,
                label=f'2n·ln(n) = {2*n*np.log(n):.0f}')
axes[1].set_title('Running average converges to E[comparisons]', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of trials')
axes[1].set_ylabel('Average comparisons')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Empirical mean: {comparisons.mean():.1f}')
print(f'Theoretical E:  {2*n*np.log(n):.1f} (2n·ln(n))')
print(f'Std deviation:  {comparisons.std():.1f}')
print(f'Min: {comparisons.min()}, Max: {comparisons.max()}')

### Probability intuition: the law of large numbers in action

In [ ]:
# Simulate: estimate π using Monte Carlo (random points in a square)
np.random.seed(42)
n_points = 5000

x = np.random.uniform(-1, 1, n_points)
y = np.random.uniform(-1, 1, n_points)
inside = x**2 + y**2 <= 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Visual: points inside/outside circle
axes[0].scatter(x[inside], y[inside], s=1, color='#6366f1', alpha=0.5)
axes[0].scatter(x[~inside], y[~inside], s=1, color='#f43f5e', alpha=0.3)
theta = np.linspace(0, 2*np.pi, 100)
axes[0].plot(np.cos(theta), np.sin(theta), 'k-', linewidth=1)
axes[0].set_aspect('equal')
axes[0].set_title(f'Monte Carlo π ≈ {4*inside.sum()/n_points:.4f}', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Convergence: π estimate improves with more points
estimates = [4 * inside[:k].sum() / k for k in range(1, n_points + 1)]
axes[1].plot(estimates, color='#6366f1', linewidth=0.5, alpha=0.7)
axes[1].axhline(np.pi, color='#10b981', linestyle='--', linewidth=2, label=f'π = {np.pi:.4f}')
axes[1].set_xlabel('Number of random points')
axes[1].set_ylabel('Estimated π')
axes[1].set_title('Convergence: more samples → better estimate', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Interactive Experiment

### QuickSort behavior explorer

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

@interact(
    n=IntSlider(value=100, min=20, max=500, step=20, description='Array size'),
    trials=IntSlider(value=200, min=50, max=1000, step=50, description='Trials'),
    input_type=Dropdown(options=['sorted', 'reversed', 'random'], value='sorted', description='Input')
)
def quicksort_explorer(n, trials, input_type):
    if input_type == 'sorted':
        arr = list(range(n))
    elif input_type == 'reversed':
        arr = list(range(n, 0, -1))
    else:
        arr = list(np.random.permutation(n))
    
    # Run Monte Carlo
    counts = np.array([quicksort_count(arr, random_pivot=True) for _ in range(trials)])
    worst = quicksort_count(arr, random_pivot=False)
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(counts, bins=30, color='#6366f1', alpha=0.7, edgecolor='white', density=True)
    ax.axvline(counts.mean(), color='#f97316', linewidth=2, linestyle='--',
               label=f'Random avg: {counts.mean():.0f}')
    ax.axvline(worst, color='#f43f5e', linewidth=2, linestyle=':',
               label=f'Deterministic: {worst}')
    ax.axvline(2*n*np.log(n), color='#10b981', linewidth=2, linestyle='-.',
               label=f'2n·ln(n): {2*n*np.log(n):.0f}')
    
    ax.set_title(f'QuickSort comparisons (n={n}, {input_type} input, {trials} trials)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Comparisons')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Monte Carlo convergence speed

In [ ]:
from ipywidgets import FloatSlider

@interact(
    n_samples=IntSlider(value=1000, min=100, max=10000, step=100, description='Samples')
)
def convergence_demo(n_samples):
    x = np.random.uniform(-1, 1, n_samples)
    y = np.random.uniform(-1, 1, n_samples)
    inside = x**2 + y**2 <= 1
    
    estimates = np.array([4 * inside[:k].sum() / k for k in range(1, n_samples + 1)])
    
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(estimates, color='#6366f1', linewidth=0.8, alpha=0.7)
    ax.axhline(np.pi, color='#10b981', linestyle='--', linewidth=2, label=f'π = {np.pi:.6f}')
    ax.set_title(f'π estimate after {n_samples} samples: {estimates[-1]:.6f} (error: {abs(estimates[-1]-np.pi):.6f})',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Samples')
    ax.set_ylabel('π estimate')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Tool Exploration

Understanding NumPy's random number generation.

In [ ]:
# Random number generation is an object
rng = np.random.default_rng(seed=42)
print(f'RNG type: {type(rng)}')
print(f'Uniform: {rng.uniform(0, 1, 5)}')
print(f'Integers: {rng.integers(0, 10, 5)}')
print(f'Normal: {rng.normal(0, 1, 5)}')
print()

# Reproducibility: same seed → same "random" numbers
rng1 = np.random.default_rng(seed=123)
rng2 = np.random.default_rng(seed=123)
print(f'Same seed produces same sequence:')
print(f'  rng1: {rng1.uniform(0, 1, 5)}')
print(f'  rng2: {rng2.uniform(0, 1, 5)}')
print()

# Vectorized sampling
samples = rng.uniform(0, 1, 100000)
print(f'100K samples - mean: {samples.mean():.4f}, std: {samples.std():.4f}')
print(f'Expected: mean=0.5, std={1/np.sqrt(12):.4f}')
print()

# Shuffling and permutation
arr = np.arange(10)
print(f'Original: {arr}')
rng.shuffle(arr)
print(f'Shuffled: {arr}')
print(f'Random permutation: {rng.permutation(10)}')

## 7. Reflection Questions

1. In the QuickSort explorer, try all three input types (sorted, reversed, random). Does the **randomized** version care which input you give it? Why?

2. The Monte Carlo π estimate converges to π as samples increase. How does the **error** decrease with the number of samples? (Hint: plot the error vs. √n)

3. If randomized QuickSort doesn't guarantee O(n log n) on every run, why do we call it an "O(n log n) algorithm"? What exactly does the Big-O refer to here?

4. Setting a random seed makes results **reproducible**. Doesn't that defeat the purpose of randomness? When is reproducibility useful vs. harmful?

5. Think about hash tables: they use hash functions (deterministic) to distribute keys. What happens when many keys hash to the same slot? How does **universal hashing** (random hash function selection) solve this — and how is it similar to random pivot selection in QuickSort?

---

*You now understand randomness. Finally: face the combinatorial explosion. → `Combinatorics_and_Explosions/`*

## 🔬 Break-It Lab
(Exercises merged from earlier structure)

In [ ]:
import numpy as np
from math101.testing import check_answer, check_complexity, difficulty_banner, score_report
from math101.algorithms import quicksort_count
from math101.tracker import record_score

results = []

## 🟢 Warm-Up: Recall & Predict

## 🟡 Challenge: Apply & Analyze

## 🔴 Boss Level: Implement From Scratch

## 📊 Results

In [ ]:
score_report(results)
record_score('Randomness_and_Expectation', 'exercises', sum(results), len(results))

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*